In [35]:
import pandas as pd
import numpy as np
# Human eligible data
human_prs = pd.read_csv('../datasets/processed/human_eligible_prs.csv')
human_prs= human_prs.rename(columns={"html_url": "pr_url"})
human_prs= human_prs.drop(columns=['Unnamed: 0'])

human_commits = pd.read_csv('../datasets/processed/human_eligible_commits.csv')
human_commit_details = pd.read_csv('../datasets/processed/human_eligible_commit_details.csv')
human_reviews = pd.read_csv('../datasets/processed/human_eligible_reviews.csv')
human_review_comments = pd.read_csv('../datasets/processed/human_eligible_inline_reviews.csv')
human_comments = pd.read_csv('../datasets/processed/human_eligible_comments.csv')

#AI eligible data
ai_prs = pd.read_csv('../datasets/processed/ai_eligible_prs.csv')
ai_prs = ai_prs.rename(columns={"html_url": "pr_url", "user": "user.login"})

ai_commits = pd.read_csv('../datasets/processed/ai_eligible_fetched_commits.csv')
ai_commit_details = pd.read_csv('../datasets/processed/ai_eligible_commit_details.csv')
ai_reviews = pd.read_csv('../datasets/processed/ai_eligible_reviews.csv')
ai_review_comments = pd.read_csv('../datasets/processed/ai_eligible_review_comments.csv')
ai_comments = pd.read_csv('../datasets/processed/ai_eligible_comments.csv')
builds = pd.read_csv('../datasets/processed/eligible_pr_failure_rates.csv')


In [36]:

# Keep only PRs from repositories that contain both human and AI PRs.
human_prs['repo_full_name'] = human_prs['pr_url'].str.extract(
    r'^https://github\.com/([^/]+/[^/]+)/pull/\d+'
)[0]
ai_prs['repo_full_name'] = ai_prs['repo_url'].str.extract(
    r'^https://api\.github\.com/repos/([^/]+/[^/]+)$'
)[0]

mixed_repo_names = set(human_prs['repo_full_name'].dropna()) & set(ai_prs['repo_full_name'].dropna())

ai_prs = ai_prs[ai_prs['repo_full_name'].isin(mixed_repo_names)].copy()
human_prs = human_prs[human_prs['repo_full_name'].isin(mixed_repo_names)].copy()

good_repos = ai_prs[['repo_url']].drop_duplicates().sort_values('repo_url').reset_index(drop=True)

allowed_human_pr_urls = set(human_prs['pr_url'].dropna())
allowed_ai_pr_ids = set(ai_prs['pr_id'].dropna())
allowed_ai_pr_urls = set(ai_prs['pr_url'].dropna())
allowed_pr_urls = allowed_human_pr_urls | allowed_ai_pr_urls


In [37]:
# combine APRs and HPRs
human_commits = human_commits[human_commits['pr_url'].isin(allowed_human_pr_urls)].copy()
human_commit_details = human_commit_details[human_commit_details['pr_url'].isin(allowed_human_pr_urls)].copy()
human_reviews = human_reviews[human_reviews['pr_url'].isin(allowed_human_pr_urls)].copy()
human_review_comments = human_review_comments[human_review_comments['pr_url'].isin(allowed_human_pr_urls)].copy()
human_comments = human_comments[human_comments['pr_url'].isin(allowed_human_pr_urls)].copy()

ai_commits = ai_commits[ai_commits['pr_id'].isin(allowed_ai_pr_ids)].copy()
ai_commit_details = ai_commit_details[ai_commit_details['pr_id'].isin(allowed_ai_pr_ids)].copy()
ai_reviews = ai_reviews[ai_reviews['pr_id'].isin(allowed_ai_pr_ids)].copy()
ai_review_comments = ai_review_comments[
    ai_review_comments['pull_request_url'].isin(
        allowed_ai_pr_urls
        | {
            url.replace('https://github.com/', 'https://api.github.com/repos/').replace('/pull/', '/pulls/')
            for url in allowed_ai_pr_urls
        }
    )
].copy()
ai_comments = ai_comments[ai_comments['pr_id'].isin(allowed_ai_pr_ids)].copy()
builds = builds[builds['pr_url'].isin(allowed_pr_urls)].copy()

human_prs = human_prs.drop(columns=['repo_full_name'])
ai_prs = ai_prs.drop(columns=['repo_full_name'])
print(f'ai prs count {len(ai_prs)}')
print(f'human prs count {len(human_prs)}')

ai prs count 264
human prs count 793


# Replace pr_id with pr_url in ai tables

In [38]:
ai_pr_lookup = ai_prs[['pr_id', 'pr_url']].drop_duplicates()

ai_commits = ai_commits.drop(columns=['pr_id'])

if 'pull_request_url' in ai_commit_details.columns:
    ai_commit_details['pr_url'] = (
        ai_commit_details['pull_request_url']
        .str.replace('https://api.github.com/repos/', 'https://github.com/', regex=False)
        .str.replace('/pulls/', '/pull/', regex=False)
    )
else:
    ai_commit_details = ai_commit_details.merge(ai_pr_lookup, on='pr_id', how='left')

ai_commit_details = ai_commit_details.drop(columns=['pr_id'])

ai_reviews = ai_reviews.merge(ai_pr_lookup, on='pr_id', how='left')
ai_reviews = ai_reviews.drop(columns=['pr_id'])

ai_review_comments['pr_url'] = (
    ai_review_comments['pull_request_url']
    .str.replace('https://api.github.com/repos/', 'https://github.com/', regex=False)
    .str.replace('/pulls/', '/pull/', regex=False)
)

ai_comments = ai_comments.merge(ai_pr_lookup, on='pr_id', how='left')
ai_comments = ai_comments.drop(columns=['pr_id'])

ai_prs = ai_prs.drop(columns=['pr_id'])


# Sampling

In [39]:
# Sample human PRs before unifying human and AI data.
RANDOM_SEED = 42
SAMPLE_FRACTION = 0.30

human_prs_for_sampling = human_prs.assign(
    repo_url=human_prs["pr_url"].str.extract(r"(https://github\.com/[^/]+/[^/]+)/pull/")[0]
)

human_count_per_repo = human_prs_for_sampling.groupby("repo_url")["pr_url"].transform("size")

human_multi = human_prs_for_sampling[human_count_per_repo > 1]
if human_multi.empty:
    human_multi_sampled = human_multi
else:
    human_multi_sampled = (
        human_multi
        .groupby("repo_url", group_keys=False)
        .sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED, replace=False)
    )

human_single = human_prs_for_sampling[human_count_per_repo == 1]

human_prs = (
    pd.concat([human_multi_sampled, human_single], ignore_index=True)
    .drop(columns=["repo_url"])
)

sampled_human_pr_urls = set(human_prs["pr_url"].dropna())
selected_pr_urls = sampled_human_pr_urls | allowed_ai_pr_urls

human_commits = human_commits[human_commits["pr_url"].isin(sampled_human_pr_urls)].copy()
human_commit_details = human_commit_details[human_commit_details["pr_url"].isin(sampled_human_pr_urls)].copy()
human_reviews = human_reviews[human_reviews["pr_url"].isin(sampled_human_pr_urls)].copy()
human_review_comments = human_review_comments[human_review_comments["pr_url"].isin(sampled_human_pr_urls)].copy()
human_comments = human_comments[human_comments["pr_url"].isin(sampled_human_pr_urls)].copy()
builds = builds[builds["pr_url"].isin(selected_pr_urls)].copy()

allowed_human_pr_urls = sampled_human_pr_urls
allowed_pr_urls = selected_pr_urls

print(f"sampled human prs {len(human_prs)}, ai prs {len(ai_prs)}, selected prs {len(selected_pr_urls)}")


sampled human prs 258, ai prs 264, selected prs 522


# Unify data

- Unify the sampled human PRs with all eligible AI PRs
- Keep side tables aligned with the sampled PR set

In [40]:
# Keep only shared columns after aligning equivalent human/ai field names.
def unify_tables(human_df, ai_df, ai_renames=None, human_renames=None):
    human = human_df.rename(columns=human_renames or {}).copy()
    ai = ai_df.rename(columns=ai_renames or {}).copy()

    human['pr_type'] = 'human'
    human['agent'] = pd.NA
    ai['pr_type'] = 'ai'

    common_columns = [col for col in human.columns if col in ai.columns]
    return pd.concat([ ai[common_columns], human[common_columns]], ignore_index=True)


prs = unify_tables(human_prs, ai_prs)
commits = unify_tables(human_commits, ai_commits)
commit_details = unify_tables(human_commit_details, ai_commit_details)

reviews = unify_tables(
    human_reviews,
    ai_reviews,
    ai_renames={'user': 'user.login', 'user_type': 'user.type'},
)
review_comments = unify_tables(
    human_review_comments,
    ai_review_comments,
    ai_renames={'user': 'user.login', 'user_type': 'user.type'},
)
comments = unify_tables(
    human_comments,
    ai_comments,
    ai_renames={'user': 'user.login', 'user_id': 'user.id', 'user_type': 'user.type'},
)

print(f"PRs: {len(prs)}, human prs {len(prs[prs['pr_type']=='human'])}, ai prs {len(prs[prs['pr_type']=='ai'])}, Commits: {len(commits)}, Commit Details: {len(commit_details)}, Reviews: {len(reviews)}, Review Comments: {len(review_comments)}, Comments: {len(comments)}, Builds: {len(builds)}")


PRs: 522, human prs 258, ai prs 264, Commits: 2242, Commit Details: 14690, Reviews: 750, Review Comments: 649, Comments: 878, Builds: 522


In [41]:
selected_pr_urls = set(prs["pr_url"])

tables_with_pr_urls = {
    'commits': commits,
    'commit_details': commit_details,
    'reviews': reviews,
    'review_comments': review_comments,
    'comments': comments,
    'builds': builds,
}

for table_name, table in tables_with_pr_urls.items():
    extra_pr_urls = set(table["pr_url"].dropna()) - selected_pr_urls
    assert not extra_pr_urls, f"{table_name} contains PRs outside the sampled set"


In [42]:
prs.to_csv('../datasets/processed/rq1/all_eligible_prs.csv', index=False)
commits.to_csv('../datasets/processed/rq1/all_eligible_commits.csv', index=False)


# User type recognizer

In [43]:
AI_AGENT_KEYWORDS = (
    "cursor",
    "copilot",
    "devin-ai-integration",
    "claude",
    "codex",
)


def classify_user_roles(
    df: pd.DataFrame,
    user_col: str = "user",
    user_type_col: str = "user_type",
    bot_value: str = "Bot",
    ai_keywords=AI_AGENT_KEYWORDS,
) -> pd.DataFrame:
    """
    Adds boolean flags:
        - is_bot   (Bot OR [bot]) AND NOT AI keyword
        - is_ai    (login contains AI keyword)
        - is_human (has a real user login and is neither bot nor AI)
    """
    out = df.copy()

    if user_col not in out.columns:
        out[user_col] = ""
    if user_type_col not in out.columns:
        out[user_type_col] = ""

    user_login = out[user_col].fillna("").astype(str)
    user_login_lower = user_login.str.lower()
    user_type = out[user_type_col].fillna("").astype(str)

    has_user = user_login.ne("")

    is_ai = has_user & user_login_lower.apply(
        lambda x: any(k in x for k in ai_keywords)
    )

    is_bot_base = has_user & (
        user_type.str.lower().eq(bot_value.lower())
        | user_login_lower.str.contains(r"\[bot\]", regex=True, na=False)
    )
    is_bot = is_bot_base & (~is_ai)

    is_human = has_user & (~is_bot) & (~is_ai)

    return out.assign(
        is_bot=is_bot,
        is_ai=is_ai,
        is_human=is_human,
    )


# First and Revision commits

In [44]:

first_commits = prs[['pr_url','pr_type','first_commit_sha']]
first_commits


revision_commits = commits.merge(
    first_commits, 
    left_on = ['pr_url','pr_type','sha'],
    right_on=['pr_url','pr_type','first_commit_sha'], 
    how='left_anti', 

)


first_commit_details = commit_details.merge(
    first_commits,
    left_on=['pr_url', 'sha'],
    right_on=['pr_url', 'first_commit_sha'],
    how='inner'
)
revision_commit_details = commit_details.merge(
    first_commits, 
    left_on = ['pr_url','sha'],
    right_on=['pr_url','first_commit_sha'], 
    how='left_anti', 

)
# print len of revision_commits and first_commits
print(f"First Commits: {len(first_commits)}, Revision Commits: {len(revision_commits)}")
print(f"First Commit Details: {len(first_commit_details)}, Revision Commit Details: {len(revision_commit_details)}")

First Commits: 522, Revision Commits: 1720
First Commit Details: 2953, Revision Commit Details: 11737


# Number and type of reviewers 

- Reviewer is the person who (comment, approve review, request changes)
- Number of reviewers are the unique users in review data
- exclude dismissed status cuz it is not review operation
- I didn't select reviewers from unique users in review_comments cuz the reviewer does not only make comments,

In [45]:
reviews_user_type = classify_user_roles(
    reviews,
    user_col='user.login',
    user_type_col='user.type'
)

number_of_reviewers = (
    reviews_user_type
    .groupby([ "pr_url",], as_index=False)
    .agg(
        # total unique reviewers (any type)
        reviewers=("user.login", lambda s: s.dropna().nunique()),

        # unique reviewers by your rules
        ai=("user.login", lambda s: s[reviews_user_type.loc[s.index, "is_ai"]].dropna().nunique()),
        human=("user.login", lambda s: s[reviews_user_type.loc[s.index, "is_human"]].dropna().nunique()),
        bot=("user.login", lambda s: s[reviews_user_type.loc[s.index, "is_bot"]].dropna().nunique()),
    )
)


# Per PR
number_of_reviewers = prs[['pr_url']].merge(number_of_reviewers, on='pr_url', how='left').fillna(0).astype({ 'reviewers': 'int', 'ai': 'int', 'human': 'int', 'bot': 'int' })
# fill reviewers, 
number_of_reviewers

,pr_url,reviewers,ai,human,bot
0,https://github.com/mlflow/mlflow/pull/16658,2,0,2,0
1,https://github.com/nrwl/nx/pull/31380,4,1,2,1
2,https://github.com/modelcontextprotocol/inspec...,1,0,1,0
3,https://github.com/sugyan/claude-code-webui/pu...,1,1,0,0
4,https://github.com/sugyan/claude-code-webui/pu...,0,0,0,0
...,...,...,...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,1,0,0,1
518,https://github.com/saturday06/VRM-Addon-for-Bl...,0,0,0,0
519,https://github.com/shugo/textbringer/pull/125,0,0,0,0
520,https://github.com/tegnike/aituber-kit/pull/402,0,0,0,0


# Number of review comments

- Number of review comments = number of inline comments

In [46]:
review_comments_user_type = classify_user_roles(
    review_comments,
    user_col='user.login',
    user_type_col='user.type'
)

number_of_review_comments = (
    review_comments_user_type
    .groupby(['pr_url', 'pr_type'], as_index=False)
    .agg(
        review_comments=('id', 'nunique'),
        ai=('id', lambda s: s[review_comments_user_type.loc[s.index, 'is_ai']].nunique()),
        human=('id', lambda s: s[review_comments_user_type.loc[s.index, 'is_human']].nunique()),
        bot=('id', lambda s: s[review_comments_user_type.loc[s.index, 'is_bot']].nunique()),
    )
)

number_of_review_comments = number_of_review_comments[
    ['pr_url',  'review_comments', 'ai', 'human', 'bot']
]


number_of_review_comments = prs[['pr_url']].merge(number_of_review_comments, on='pr_url', how='left').fillna(0).astype({ 'review_comments': 'int', 'ai': 'int', 'human': 'int', 'bot': 'int' })

number_of_review_comments.sort_values('review_comments', ascending=False)
 


,pr_url,review_comments,ai,human,bot
9,https://github.com/foxminchan/BookWorm/pull/53,72,26,46,0
346,https://github.com/hyperlight-dev/hyperlight/p...,43,0,43,0
262,https://github.com/AgentOps-AI/agentops/pull/603,30,0,0,30
11,https://github.com/hyperlight-dev/hyperlight/p...,27,13,14,0
12,https://github.com/hyperlight-dev/hyperlight/p...,15,6,9,0
...,...,...,...,...,...
195,https://github.com/cloudwalk/stratus/pull/1968,0,0,0,0
194,https://github.com/crewAIInc/crewAI/pull/2795,0,0,0,0
193,https://github.com/tegnike/aituber-kit/pull/358,0,0,0,0
191,https://github.com/digitaldemocracy2030/koucho...,0,0,0,0


# Number of comments

In [47]:
comments_user_type = classify_user_roles(
    comments,
    user_col='user.login',
    user_type_col='user.type'
)

number_of_comments = (
    comments_user_type
    .groupby(['pr_url'], as_index=False)
    .agg(
        comments=('id', 'nunique'),
        ai=('id', lambda s: s[comments_user_type.loc[s.index, 'is_ai']].nunique()),
        human=('id', lambda s: s[comments_user_type.loc[s.index, 'is_human']].nunique()),
        bot=('id', lambda s: s[comments_user_type.loc[s.index, 'is_bot']].nunique()),
    )
)

number_of_comments = number_of_comments[
    ['pr_url',  'comments', 'ai', 'human', 'bot']
]


number_of_comments = prs[['pr_url']].merge(number_of_comments, on='pr_url', how='left').fillna(0).astype({ 'comments': 'int', 'ai': 'int', 'human': 'int', 'bot': 'int' })

number_of_comments.sort_values('comments', ascending=False)


,pr_url,comments,ai,human,bot
9,https://github.com/foxminchan/BookWorm/pull/53,30,13,16,1
11,https://github.com/hyperlight-dev/hyperlight/p...,27,14,13,0
12,https://github.com/hyperlight-dev/hyperlight/p...,17,9,8,0
136,https://github.com/trycompai/comp/pull/1072,14,0,2,12
484,https://github.com/whitphx/stlite/pull/1365,13,0,1,12
...,...,...,...,...,...
94,https://github.com/sandrohanea/whisper.net/pul...,0,0,0,0
91,https://github.com/sandrohanea/whisper.net/pul...,0,0,0,0
89,https://github.com/bruin-data/bruin/pull/735,0,0,0,0
88,https://github.com/go-vikunja/vikunja/pull/1178,0,0,0,0


# Number of commits

In [48]:
commits_user_type = classify_user_roles(
    commits,
    user_col='author.login',
    user_type_col='author.type'
)

number_of_commits = (
    commits_user_type
    .groupby(['pr_url'], as_index=False)
    .agg(
        commits=('sha', 'nunique'),
        ai=('sha', lambda s: s[commits_user_type.loc[s.index, 'is_ai']].nunique()),
        human=('sha', lambda s: s[commits_user_type.loc[s.index, 'is_human']].nunique()),
        bot=('sha', lambda s: s[commits_user_type.loc[s.index, 'is_bot']].nunique()),
    )
)

number_of_commits = number_of_commits[
    ['pr_url',  'commits', 'ai', 'human', 'bot']
]



number_of_commits = prs[['pr_url']].merge(number_of_commits, on='pr_url', how='left').fillna(0).astype({ 'commits': 'int', 'ai': 'int', 'human': 'int', 'bot': 'int' })

number_of_commits.sort_values('commits', ascending=False)


,pr_url,commits,ai,human,bot
381,https://github.com/mlflow/mlflow/pull/16719,105,8,94,3
265,https://github.com/AMICI-dev/AMICI/pull/2839,44,0,44,0
371,https://github.com/mlflow/mlflow/pull/16439,44,0,44,0
362,https://github.com/leoafarias/fvm/pull/858,42,0,42,0
372,https://github.com/mlflow/mlflow/pull/16564,39,6,32,1
...,...,...,...,...,...
238,https://github.com/lingodotdev/lingo.dev/pull/695,1,1,0,0
236,https://github.com/pdfme/pdfme/pull/866,1,1,0,0
229,https://github.com/marimo-team/marimo/pull/4012,1,1,0,0
227,https://github.com/novuhq/novu/pull/8365,1,1,0,0


# Number of Revisions

In [49]:
revision_commits_user_type = classify_user_roles(
    revision_commits,
    user_col='author.login',
    user_type_col='author.type'
)

number_of_revision_commits = (
    revision_commits_user_type
    .groupby(['pr_url'], as_index=False)
    .agg(
        revision=('sha', 'nunique'),
        ai=('sha', lambda s: s[revision_commits_user_type.loc[s.index, 'is_ai']].nunique()),
        human=('sha', lambda s: s[revision_commits_user_type.loc[s.index, 'is_human']].nunique()),
        bot=('sha', lambda s: s[revision_commits_user_type.loc[s.index, 'is_bot']].nunique()),
    )
)

number_of_revision_commits = number_of_revision_commits[
    ['pr_url',  'revision', 'ai', 'human', 'bot']
]
number_of_revision_commits = prs[['pr_url']].merge(number_of_revision_commits, on='pr_url', how='left').fillna(0).astype({ 'revision': 'int', 'ai': 'int', 'human': 'int', 'bot': 'int' })

number_of_revision_commits.sort_values('revision', ascending=False)


,pr_url,revision,ai,human,bot
381,https://github.com/mlflow/mlflow/pull/16719,104,8,93,3
265,https://github.com/AMICI-dev/AMICI/pull/2839,43,0,43,0
371,https://github.com/mlflow/mlflow/pull/16439,43,0,43,0
362,https://github.com/leoafarias/fvm/pull/858,41,0,41,0
372,https://github.com/mlflow/mlflow/pull/16564,38,6,31,1
...,...,...,...,...,...
238,https://github.com/lingodotdev/lingo.dev/pull/695,0,0,0,0
236,https://github.com/pdfme/pdfme/pull/866,0,0,0,0
229,https://github.com/marimo-team/marimo/pull/4012,0,0,0,0
227,https://github.com/novuhq/novu/pull/8365,0,0,0,0


# Number of changed files

## First commit

### Workflow files

In [50]:
# extract number of changed workflow files in the first commit


initial_commit_workflow_files = (
    first_commit_details
        .groupby(['pr_url', 'sha'], as_index=False)
        .agg(
            initial_workflow_files=(
                'filename',
                lambda s: (
                    s.str.startswith('.github/workflows/') &
                    s.str.endswith(('.yml', '.yaml'))
                ).sum()
            )
        )
)

initial_commit_workflow_files = initial_commit_workflow_files[['pr_url','initial_workflow_files']]
initial_commit_workflow_files.sort_values('initial_workflow_files')


,pr_url,initial_workflow_files
0,https://github.com/3rdIteration/btcrecover/pul...,1
332,https://github.com/openai/openai-agents-js/pul...,1
331,https://github.com/openai/openai-agents-js/pul...,1
330,https://github.com/openai/openai-agents-js/pul...,1
329,https://github.com/openai/codex/pull/964,1
...,...,...
65,https://github.com/Stirling-Tools/Stirling-PDF...,9
306,https://github.com/novuhq/novu/pull/8639,10
381,https://github.com/pyth-network/pyth-crosschai...,11
380,https://github.com/pyth-network/pyth-crosschai...,11


### Total files

In [51]:

total_initial_commit_files_grouped = (
    first_commit_details
    .groupby(['pr_url'],as_index=False)['filename']
    .nunique()
    .rename(columns={"filename":"initial_total_changed_files"})
)[['pr_url','initial_total_changed_files']] # type: ignore
total_initial_commit_files_grouped.sort_values('initial_total_changed_files', ascending=False)

,pr_url,initial_total_changed_files
183,https://github.com/giselles-ai/giselle/pull/1330,300
245,https://github.com/leoafarias/fvm/pull/858,168
262,https://github.com/lingodotdev/lingo.dev/pull/962,119
440,https://github.com/tegnike/aituber-kit/pull/380,100
28,https://github.com/DannyMac180/meta-agent/pull/16,74
...,...,...
242,https://github.com/langfuse/langfuse/pull/8076,1
241,https://github.com/langfuse/langfuse/pull/7944,1
239,https://github.com/langfuse/langfuse/pull/7845,1
238,https://github.com/langfuse/langfuse-docs/pull...,1


## revision commits

### Workflow files

In [52]:
revision_pr_workflow_files = (
    revision_commits[['pr_url']]
    .drop_duplicates()
    .merge(
        revision_commit_details
        .dropna(subset=['filename'])
        .loc[
            lambda df: (
                df['filename'].str.startswith('.github/workflows/', na=False) &
                df['filename'].str.endswith(('.yml', '.yaml'), na=False)
            )
        ]
        .groupby(['pr_url'], as_index=False)
        .agg(
            revision_workflow_files=('filename', 'nunique')
        ),
        on=['pr_url'],
        how='left'
    )
)

revision_pr_workflow_files['revision_workflow_files'] = (
    revision_pr_workflow_files['revision_workflow_files']
    .fillna(0)
    .astype('Int64')
)

revision_pr_workflow_files = (
    prs[['pr_url']]
    .drop_duplicates()
    .merge(revision_pr_workflow_files, on='pr_url', how='left')
)

revision_pr_workflow_files['revision_workflow_files'] = (
    revision_pr_workflow_files['revision_workflow_files']
    .astype('Int64')
)

revision_pr_workflow_files.sort_values('revision_workflow_files')


,pr_url,revision_workflow_files
242,https://github.com/ensdomains/ens-app-v3/pull/938,0
163,https://github.com/reflex-dev/reflex-web/pull/...,0
312,https://github.com/cloudwalk/stratus/pull/2062,0
313,https://github.com/cloudwalk/stratus/pull/2061,0
314,https://github.com/cloudwalk/stratus/pull/2086,0
...,...,...
516,https://github.com/osori/korean-romanizer/pull/19,<NA>
518,https://github.com/saturday06/VRM-Addon-for-Bl...,<NA>
519,https://github.com/shugo/textbringer/pull/125,<NA>
520,https://github.com/tegnike/aituber-kit/pull/402,<NA>


### Total files 

In [53]:
revision_commits_total_files_change = (
    revision_commits[['pr_url']]
    .drop_duplicates()
    .merge(
        revision_commit_details
        .dropna(subset=['filename'])
        .groupby('pr_url', as_index=False)
        .agg(
            revision_total_changed_files=('filename', 'nunique')
        ),
        on='pr_url',
        how='left'
    )
)

revision_commits_total_files_change['revision_total_changed_files'] = (
    revision_commits_total_files_change['revision_total_changed_files']
    .fillna(0)
    .astype('Int64')
)

revision_commits_total_files_change = (
    prs[['pr_url']]
    .drop_duplicates()
    .merge(revision_commits_total_files_change, on='pr_url', how='left')
)

revision_commits_total_files_change['revision_total_changed_files'] = (
    revision_commits_total_files_change['revision_total_changed_files']
    .astype('Int64')
)

revision_commits_total_files_change.sort_values('revision_total_changed_files')


,pr_url,revision_total_changed_files
470,https://github.com/trycompai/comp/pull/351,0
463,https://github.com/trycompai/comp/pull/405,0
464,https://github.com/trycompai/comp/pull/352,0
417,https://github.com/pyth-network/pyth-crosschai...,1
477,https://github.com/vectordotdev/vector/pull/23318,1
...,...,...
516,https://github.com/osori/korean-romanizer/pull/19,<NA>
518,https://github.com/saturday06/VRM-Addon-for-Bl...,<NA>
519,https://github.com/shugo/textbringer/pull/125,<NA>
520,https://github.com/tegnike/aituber-kit/pull/402,<NA>


## Total file change

In [54]:


total_file_change_df = (
    commit_details
    .dropna(subset=['filename'])
    .groupby('pr_url', as_index=False)['filename']
    .nunique()
    .rename(columns={'filename': 'changed_files'})
)


# add all in one
total_file_change = (
    total_file_change_df
        .merge(initial_commit_workflow_files, on='pr_url', how='left')
        .merge(total_initial_commit_files_grouped, on='pr_url', how='left')
        .merge(revision_pr_workflow_files, on='pr_url', how='left')
        .merge(revision_commits_total_files_change, on='pr_url', how='left')
)



total_file_change.sort_values('changed_files')


,pr_url,changed_files,initial_workflow_files,initial_total_changed_files,revision_workflow_files,revision_total_changed_files
260,https://github.com/lingodotdev/lingo.dev/pull/730,1,1,1,<NA>,<NA>
258,https://github.com/lingodotdev/lingo.dev/pull/727,1,1,1,<NA>,<NA>
261,https://github.com/lingodotdev/lingo.dev/pull/732,1,1,1,<NA>,<NA>
264,https://github.com/marimo-team/marimo/pull/3474,1,1,1,<NA>,<NA>
270,https://github.com/mcpgod/cli/pull/11,1,1,1,1,1
...,...,...,...,...,...,...
201,https://github.com/hyperlight-dev/hyperlight/p...,439,2,19,0,426
12,https://github.com/AMICI-dev/AMICI/pull/2839,444,1,1,19,444
200,https://github.com/hyperlight-dev/hyperlight/p...,449,1,4,10,449
291,https://github.com/mlflow/mlflow/pull/16719,510,1,2,7,509


# Review duration

In [55]:

review_duration = pd.to_datetime(prs['closed_at']) - pd.to_datetime(prs['created_at'])
review_duration= review_duration.dt.total_seconds()


# I want to calculate review diration for each pr
prs['review_duration'] = review_duration
review_duration = prs[['pr_url', 'review_duration']]
review_duration.sort_values('review_duration', ascending=False)

,pr_url,review_duration
344,https://github.com/go-vikunja/vikunja/pull/875,23401144.0
281,https://github.com/Kiln-AI/Kiln/pull/458,12614060.0
265,https://github.com/AMICI-dev/AMICI/pull/2839,12007313.0
392,https://github.com/nrwl/nx/pull/31783,6417111.0
127,https://github.com/oven-sh/bun/pull/19763,5329149.0
...,...,...
521,https://github.com/theosanderson/taxonium/pull...,7.0
294,https://github.com/alexanderjeurissen/ranger_d...,6.0
264,https://github.com/567-labs/instructor/pull/1688,6.0
427,https://github.com/pyth-network/pyth-crosschai...,6.0


# Length of discussion

In [56]:
comments_body_length_per_pr = comments.groupby(['pr_url'])['body'].apply(lambda x: x.str.len().sum()).reset_index(name='comments_body_length')
review_comments_body_length_per_pr = review_comments.groupby(['pr_url'])['body'].apply(lambda x: x.str.len().sum()).reset_index(name='review_comments_body_length')

length_of_discussion = comments_body_length_per_pr.merge(review_comments_body_length_per_pr, on=['pr_url'], how='outer').fillna(0).astype({'comments_body_length': 'int', 'review_comments_body_length': 'int'})
length_of_discussion['length_of_discussion'] = length_of_discussion['comments_body_length'] + length_of_discussion['review_comments_body_length']

length_of_discussion = prs[['pr_url']].merge(length_of_discussion, on=['pr_url'], how='left').fillna(0).astype({'comments_body_length': 'int', 'review_comments_body_length': 'int', 'length_of_discussion': 'int'})
length_of_discussion = length_of_discussion[['pr_url', 'length_of_discussion']]
length_of_discussion.sort_values('length_of_discussion', ascending=False)

,pr_url,length_of_discussion
159,https://github.com/whitphx/stlite/pull/1312,899679
484,https://github.com/whitphx/stlite/pull/1365,762506
489,https://github.com/whitphx/stlite/pull/1351,380669
488,https://github.com/whitphx/stlite/pull/1324,380417
487,https://github.com/whitphx/stlite/pull/1346,255752
...,...,...
409,https://github.com/openvm-org/openvm/pull/1788,0
100,https://github.com/mcpgod/cli/pull/11,0
280,https://github.com/GlareDB/glaredb/pull/3663,0
279,https://github.com/GlareDB/glaredb/pull/3947,0


# Length of description

In [57]:
length_of_description = prs[['pr_url', 'body']].copy()
length_of_description['length_of_description'] = length_of_description['body'].str.len()
length_of_description = length_of_description.drop(columns=['body'])

length_of_description = prs[['pr_url']].merge(length_of_description, on='pr_url', how='left').fillna(0).astype({'length_of_description': 'int'})
length_of_description.sort_values('length_of_description', ascending=False)

,pr_url,length_of_description
5,https://github.com/giselles-ai/giselle/pull/1330,77435
343,https://github.com/giselles-ai/giselle/pull/1441,5847
374,https://github.com/mlflow/mlflow/pull/16735,5424
369,https://github.com/mlflow/mlflow/pull/16678,5349
375,https://github.com/mlflow/mlflow/pull/16626,5324
...,...,...
456,https://github.com/theopenco/llmgateway/pull/425,0
455,https://github.com/theopenco/llmgateway/pull/409,0
454,https://github.com/theopenco/llmgateway/pull/444,0
326,https://github.com/dcSpark/shinkai-local-ai-ag...,0


# Code churn

## First commit code churn

In [58]:
initial_code_churn = (
    first_commit_details
    .groupby(['pr_url', 'sha'], as_index=False)
    .agg(initial_code_churn=('changes', 'sum'))
    [['pr_url', 'initial_code_churn']]
)

initial_code_churn.sort_values('initial_code_churn', ascending=False)

,pr_url,initial_code_churn
67,https://github.com/Yacht-sh/Yacht/pull/757,23590.0
16,https://github.com/AgentOps-AI/agentops/pull/1179,23008.0
440,https://github.com/tegnike/aituber-kit/pull/380,19790.0
311,https://github.com/nrwl/nx/pull/31470,18180.0
245,https://github.com/leoafarias/fvm/pull/858,17860.0
...,...,...
332,https://github.com/openai/openai-agents-js/pul...,1.0
481,https://github.com/vectordotdev/vector/pull/23283,1.0
62,https://github.com/Stirling-Tools/Stirling-PDF...,1.0
480,https://github.com/vectordotdev/vector/pull/23281,1.0


## Revision commits code churn

In [59]:
revision_code_churn = (
    revision_commit_details
    .groupby(['pr_url', 'sha'], as_index=False)
    .agg(total_changes=('changes', 'sum'))
    .groupby('pr_url', as_index=False)
    .agg(revision_code_churn=('total_changes', 'sum'))
)

revision_code_churn = (
    prs[['pr_url']]
    .merge(revision_code_churn, on='pr_url', how='left')
)
revision_code_churn['revision_code_churn'] = revision_code_churn['revision_code_churn'].fillna(pd.NA).astype('Int64')
revision_code_churn

,pr_url,revision_code_churn
0,https://github.com/mlflow/mlflow/pull/16658,<NA>
1,https://github.com/nrwl/nx/pull/31380,658
2,https://github.com/modelcontextprotocol/inspec...,358
3,https://github.com/sugyan/claude-code-webui/pu...,1251
4,https://github.com/sugyan/claude-code-webui/pu...,<NA>
...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,75
518,https://github.com/saturday06/VRM-Addon-for-Bl...,<NA>
519,https://github.com/shugo/textbringer/pull/125,<NA>
520,https://github.com/tegnike/aituber-kit/pull/402,<NA>


## Total code churn

In [60]:
total_code_churn = (
    commit_details
    .groupby(['pr_url', 'sha'], as_index=False)
    .agg(total_changes=('changes', 'sum'))
    .groupby('pr_url', as_index=False)
    .agg(code_churn=('total_changes', 'sum'))
)

total_code_churn = (
    prs[['pr_url']]
    .merge(total_code_churn, on='pr_url', how='left')
)
total_code_churn = (
    total_code_churn
    .merge(initial_code_churn, on='pr_url', how='left')
    .merge(revision_code_churn, on='pr_url', how='left')
)

total_code_churn

,pr_url,code_churn,initial_code_churn,revision_code_churn
0,https://github.com/mlflow/mlflow/pull/16658,620.0,620.0,<NA>
1,https://github.com/nrwl/nx/pull/31380,695.0,37.0,658
2,https://github.com/modelcontextprotocol/inspec...,696.0,338.0,358
3,https://github.com/sugyan/claude-code-webui/pu...,2251.0,1000.0,1251
4,https://github.com/sugyan/claude-code-webui/pu...,9.0,9.0,<NA>
...,...,...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,134.0,59.0,75
518,https://github.com/saturday06/VRM-Addon-for-Bl...,5.0,5.0,<NA>
519,https://github.com/shugo/textbringer/pull/125,39.0,39.0,<NA>
520,https://github.com/tegnike/aituber-kit/pull/402,37.0,37.0,<NA>


## Workflow files code churn

In [61]:
workflow_code_churn = (
    commit_details
    .dropna(subset=['filename'])
    .loc[lambda df:
        df['filename'].str.startswith('.github/workflows/', na=False) &
        df['filename'].str.endswith(('.yml', '.yaml'), na=False)
    ]
    .groupby(['pr_url', 'sha'], as_index=False)
    .agg(total_changes=('changes', 'sum'))
    .groupby('pr_url', as_index=False)
    .agg(workflow_code_churn=('total_changes', 'sum'))
)

workflow_code_churn = (
    prs[['pr_url']]
    .merge(workflow_code_churn, on='pr_url', how='left')
)
workflow_code_churn['workflow_code_churn'] = workflow_code_churn['workflow_code_churn'].fillna(pd.NA).astype('Int64')
workflow_code_churn

,pr_url,workflow_code_churn
0,https://github.com/mlflow/mlflow/pull/16658,7
1,https://github.com/nrwl/nx/pull/31380,162
2,https://github.com/modelcontextprotocol/inspec...,52
3,https://github.com/sugyan/claude-code-webui/pu...,1064
4,https://github.com/sugyan/claude-code-webui/pu...,9
...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,134
518,https://github.com/saturday06/VRM-Addon-for-Bl...,5
519,https://github.com/shugo/textbringer/pull/125,39
520,https://github.com/tegnike/aituber-kit/pull/402,37


# Acceptance rate

In [62]:
#we measure PR acceptance rate as the ratio of merged to submitted prs
ai_closed_prs = ai_prs[ai_prs['state'] == 'closed']
human_closed_prs = human_prs[human_prs['state'] == 'closed']

ai_accepted_prs = (
    ai_closed_prs['merged_at'].notna().sum() / len(ai_closed_prs)
    if len(ai_closed_prs) > 0 else 0
)

human_accepted_prs = (
    human_closed_prs['merged_at'].notna().sum() / len(human_closed_prs)
    if len(human_closed_prs) > 0 else 0
)


accepted_prs =prs[['pr_url','merged_at']]
# add new column named accepted and set it to 1 if merged_at is not null, else 0
accepted_prs['accepted'] = accepted_prs['merged_at'].notna().astype(int)
accepted_prs = accepted_prs[['pr_url','accepted']]
accepted_prs

,pr_url,accepted
0,https://github.com/mlflow/mlflow/pull/16658,1
1,https://github.com/nrwl/nx/pull/31380,1
2,https://github.com/modelcontextprotocol/inspec...,1
3,https://github.com/sugyan/claude-code-webui/pu...,1
4,https://github.com/sugyan/claude-code-webui/pu...,1
...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,1
518,https://github.com/saturday06/VRM-Addon-for-Bl...,1
519,https://github.com/shugo/textbringer/pull/125,1
520,https://github.com/tegnike/aituber-kit/pull/402,1


In [63]:
workflow_failure_rate = prs[['pr_url']].merge(builds[['pr_url', 'workflow_failure_rate']], on='pr_url', how='left')

# Continuous per-PR build failure rate from completed success/failure CI builds.
build_failure_rate = prs[['pr_url']].merge(
    builds[['pr_url', 'build_failure_rate']], on='pr_url', how='left'
)

# Initial-commit failure remains a binary indicator.
initial_build_failure_rate = (
    prs[['pr_url']]
    .merge(builds[['pr_url', 'initial_commit_failed']], on='pr_url', how='left')
    .rename(columns={'initial_commit_failed': 'initial_build_failure_rate'})
)
initial_build_failure_rate['initial_build_failure_rate'] = (
    initial_build_failure_rate['initial_build_failure_rate']
    .astype('boolean')
    .astype('Float64')
)
workflow_failure_rate


,pr_url,workflow_failure_rate
0,https://github.com/mlflow/mlflow/pull/16658,0.000000
1,https://github.com/nrwl/nx/pull/31380,0.000000
2,https://github.com/modelcontextprotocol/inspec...,0.166667
3,https://github.com/sugyan/claude-code-webui/pu...,0.000000
4,https://github.com/sugyan/claude-code-webui/pu...,0.000000
...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,0.250000
518,https://github.com/saturday06/VRM-Addon-for-Bl...,0.000000
519,https://github.com/shugo/textbringer/pull/125,0.000000
520,https://github.com/tegnike/aituber-kit/pull/402,0.000000


# Combine all metrics

In [64]:
# combine all metrics into one dataframe for analysis
combined_metrics = pd.concat([
    prs[['pr_url', 'pr_type', 'agent']].set_index('pr_url'),

    number_of_reviewers.rename(columns={
        'ai': 'ai_reviewers',
        'human': 'human_reviewers',
        'bot': 'bot_reviewers',
    }).set_index('pr_url'),

    number_of_review_comments.rename(columns={
        'ai': 'ai_review_comments',
        'human': 'human_review_comments',
        'bot': 'bot_review_comments',
    }).set_index('pr_url'),

    number_of_comments.rename(columns={
        'ai': 'ai_comments',
        'human': 'human_comments',
        'bot': 'bot_comments',
    }).set_index('pr_url'),

    number_of_commits.rename(columns={
        'ai': 'ai_commits',
        'human': 'human_commits',
        'bot': 'bot_commits',
    }).set_index('pr_url'),

    number_of_revision_commits.rename(columns={
        'ai': 'ai_revision_commits',
        'human': 'human_revision_commits',
        'bot': 'bot_revision_commits',
    }).set_index('pr_url'),

    total_file_change.set_index('pr_url')[
        [
            'changed_files',
            'initial_workflow_files',
            'initial_total_changed_files',
            'revision_workflow_files',
            'revision_total_changed_files',
        ]
    ],

    review_duration.set_index('pr_url'),
    length_of_discussion.set_index('pr_url'),
    length_of_description.set_index('pr_url'),

    total_code_churn.set_index('pr_url')[
        [
            'code_churn',
            'initial_code_churn',
            'revision_code_churn',
        ]
    ],

    workflow_code_churn.set_index('pr_url'),

    accepted_prs.set_index('pr_url'),
    build_failure_rate.set_index('pr_url'),
    initial_build_failure_rate.set_index('pr_url'),
    workflow_failure_rate.set_index('pr_url'),

], axis=1).reset_index()

combined_metrics['repo_url'] = combined_metrics['pr_url'].str.extract(
    r'^(https://github\.com/[^/]+/[^/]+)/pull/\d+'
)[0]

# put repo_url as the first column
cols = combined_metrics.columns.tolist()
cols.insert(0, cols.pop(cols.index('repo_url')))
combined_metrics = combined_metrics[cols]

# group by repo_url
combined_metrics = combined_metrics.sort_values(
    ['repo_url', 'pr_url']
).reset_index(drop=True)

combined_metrics.to_csv('../datasets/processed/combined_metrics.csv', na_rep='NA', index=False)
combined_metrics


,repo_url,pr_url,pr_type,agent,reviewers,ai_reviewers,human_reviewers,bot_reviewers,review_comments,ai_review_comments,...,length_of_discussion,length_of_description,code_churn,initial_code_churn,revision_code_churn,workflow_code_churn,accepted,build_failure_rate,initial_build_failure_rate,workflow_failure_rate
0,https://github.com/3rdIteration/btcrecover,https://github.com/3rdIteration/btcrecover/pul...,ai,OpenAI_Codex,0,0,0,0,0,0,...,0,403,470.0,64.0,406,345,1,0.184295,1.0,0.759259
1,https://github.com/3rdIteration/btcrecover,https://github.com/3rdIteration/btcrecover/pul...,ai,OpenAI_Codex,0,0,0,0,0,0,...,0,342,740.0,672.0,68,64,1,0.217557,1.0,0.709677
2,https://github.com/3rdIteration/btcrecover,https://github.com/3rdIteration/btcrecover/pul...,human,<NA>,0,0,0,0,0,0,...,0,471,60.0,56.0,4,12,1,0.236842,1.0,0.400000
3,https://github.com/567-labs/instructor,https://github.com/567-labs/instructor/pull/1650,ai,OpenAI_Codex,1,0,0,1,0,0,...,628,1237,7.0,7.0,<NA>,7,0,0.125000,1.0,0.166667
4,https://github.com/567-labs/instructor,https://github.com/567-labs/instructor/pull/1688,human,<NA>,1,0,0,1,0,0,...,5228,3204,113.0,35.0,78,113,1,0.117647,0.0,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517,https://github.com/wvlet/wvlet,https://github.com/wvlet/wvlet/pull/1013,human,<NA>,0,0,0,0,0,0,...,134,793,3.0,3.0,<NA>,3,1,0.000000,0.0,0.000000
518,https://github.com/wvlet/wvlet,https://github.com/wvlet/wvlet/pull/991,ai,Claude_Code,2,0,1,1,8,0,...,8932,2199,536.0,346.0,190,240,1,0.097561,1.0,0.117647
519,https://github.com/yamadashy/repomix,https://github.com/yamadashy/repomix/pull/541,ai,Devin,1,1,0,0,0,0,...,3600,666,26.0,26.0,<NA>,22,0,0.000000,0.0,0.000000
520,https://github.com/yamadashy/repomix,https://github.com/yamadashy/repomix/pull/543,ai,Devin,1,1,0,0,0,0,...,3553,504,12.0,10.0,2,4,1,0.000000,0.0,0.000000


# Workflow commits

In [65]:
# commits where all changed files are workflow files only
workflow_commit_keys = (
    commit_details
    .dropna(subset=["filename"])
    .assign(
        is_workflow_file=lambda df: (
            df["filename"].str.startswith(".github/workflows/", na=False)
            & df["filename"].str.endswith((".yml", ".yaml"), na=False)
        )
    )
    .groupby(["pr_url", "pr_type", "sha"], as_index=False)
    .agg(
        changed_files=("filename", "nunique"),
        workflow_files=("is_workflow_file", "sum"),
        non_workflow_files=("is_workflow_file", lambda s: (~s).sum()),
    )
    .loc[lambda df: (df["workflow_files"] > 0) & (df["non_workflow_files"] == 0)]
    .drop(columns=["non_workflow_files"])
)

workflow_commits = (
    commits
    .merge(
        workflow_commit_keys,
        on=["pr_url", "pr_type", "sha"],
        how="inner",
    )
    .drop_duplicates(["pr_url", "pr_type", "sha"])
    .sort_values(["pr_type", "pr_url", "sha"])
    .reset_index(drop=True)
)

print(f"Workflow-only commits: {len(workflow_commits)}")
workflow_commits

workflow_commit_details = (
    commit_details
    .merge(
        workflow_commit_keys[["pr_url", "pr_type", "sha"]],
        on=["pr_url", "pr_type", "sha"],
        how="inner",
    )
    .sort_values(["pr_type", "pr_url", "sha", "filename"])
    .reset_index(drop=True)
)

workflow_commit_details




Workflow-only commits: 880


,pr_url,sha,filename,status,additions,deletions,changes,patch,pr_type
0,https://github.com/3rdIteration/btcrecover/pul...,25009220f19cf26109d9bec8296f4c9d714b09fe,.github/workflows/termux-tests.yml,modified,2.0,2.0,4.0,"@@ -47,7 +47,7 @@ jobs:\n run: chown -...",ai
1,https://github.com/3rdIteration/btcrecover/pul...,d71c76159eda3dfd1fa0aa11ab3d537fc658e16d,.github/workflows/termux-tests.yml,modified,2.0,2.0,4.0,"@@ -16,7 +16,7 @@ jobs:\n - runner: ...",ai
2,https://github.com/3rdIteration/btcrecover/pul...,aff0a44f74cd25cb4a348a7d8e5cc296c7de5ec5,.github/workflows/PoCL-tests.yml,modified,2.0,0.0,2.0,"@@ -3,6 +3,8 @@ name: OpenCL PoCL Tests\n on:\...",ai
3,https://github.com/567-labs/instructor/pull/1650,15cf33dd5964c2ae62c99d6dd8f56b9296a91f02,.github/workflows/test_docs.yml,modified,1.0,6.0,7.0,"@@ -3,7 +3,7 @@ on:\n pull_request:\n push...",ai
4,https://github.com/AMICI-dev/AMICI/pull/2795,5aeccd699e6a0b67e54a1e292df4c3539c2cbc00,.github/workflows/test_sbml_semantic_test_suit...,modified,13.0,2.0,15.0,"@@ -21,7 +21,18 @@ jobs:\n name: SBML Sema...",ai
...,...,...,...,...,...,...,...,...,...
1177,https://github.com/whitphx/stlite/pull/1464,c0bec0cdd1e843c88772d06ff08756be7e42bc34,.github/workflows/postbuild.yml,modified,10.0,0.0,10.0,"@@ -172,9 +172,14 @@ jobs:\n - uses: actio...",human
1178,https://github.com/wvlet/wvlet/pull/1011,9b56113cd8ee87ce4dc7f53137629cea077cd345,.github/workflows/python-publish.yml,modified,1.0,10.0,11.0,"@@ -168,18 +168,9 @@ jobs:\n pip i...",human
1179,https://github.com/wvlet/wvlet/pull/1012,f3516c780e761f9a4ec9138d53c1a42e8d134741,.github/workflows/python-publish.yml,modified,4.0,0.0,4.0,"@@ -18,6 +18,8 @@ jobs:\n runs-on: ubuntu-...",human
1180,https://github.com/wvlet/wvlet/pull/1013,d93d4d0697f0e11cd654f44f3be7f8cd4d19cbb5,.github/workflows/python-publish.yml,modified,2.0,1.0,3.0,"@@ -206,4 +206,5 @@ jobs:\n - name: Publ...",human


In [66]:
prs

,pr_url,first_commit_sha,state,title,user.login,body,created_at,closed_at,merged_at,pr_type,agent,review_duration
0,https://github.com/mlflow/mlflow/pull/16658,9325e444ec074e4c1b636b52c6492217eec7bf23,closed,Add function signature breaking change detector,harupy,<details><summary>&#x1F6E0 DevTools &#x1F6E0</...,2025-07-09T05:35:26Z,2025-07-11T05:13:35Z,2025-07-11T05:13:35Z,ai,Claude_Code,171489.0
1,https://github.com/nrwl/nx/pull/31380,5943104753be2820df8bac1fe123644bd3f461e6,closed,chore(repo): setup claude code,FrozenPandaz,## Summary\n\nThis PR integrates Claude Code A...,2025-05-29T02:21:07Z,2025-05-30T22:14:12Z,2025-05-30T22:14:12Z,ai,Claude_Code,157985.0
2,https://github.com/modelcontextprotocol/inspec...,b70cf563a867efc4c9fa376aa02e991ce4360867,closed,feat: add version management scripts and CI check,jerome3o-anthropic,## Summary\n\nThis PR adds automated version m...,2025-06-14T18:53:08Z,2025-06-17T20:31:21Z,2025-06-17T20:31:21Z,ai,Claude_Code,265093.0
3,https://github.com/sugyan/claude-code-webui/pu...,101f227d028edb0422e7d048093ee902fb73bf26,closed,Implement automated demo recording CI/CD pipel...,sugyan,## Type of Change\n\n- [ ] 🐛 `bug` - Bug fix (...,2025-06-22T02:21:35Z,2025-06-22T13:55:42Z,2025-06-22T13:55:42Z,ai,Claude_Code,41647.0
4,https://github.com/sugyan/claude-code-webui/pu...,037cc75189cba847cbe6aba10fd29df7a8b4505d,closed,fix: replace unreliable FedericoCarboni/setup-...,sugyan,## Type of Change\n\n- [x] 🐛 `bug` - Bug fix (...,2025-07-04T00:33:24Z,2025-07-04T00:39:34Z,2025-07-04T00:39:34Z,ai,Claude_Code,370.0
...,...,...,...,...,...,...,...,...,...,...,...,...
517,https://github.com/pdfme/pdfme/pull/1101,bbdedec7efef675c6de34e48001d5a5f67ed41ad,closed,Add claude GitHub actions 1751256350061,hand-dot,NaN,2025-06-30T04:07:20Z,2025-06-30T04:08:21Z,2025-06-30T04:08:21Z,human,<NA>,61.0
518,https://github.com/saturday06/VRM-Addon-for-Bl...,fbcaee149415b19e7d5631e89710ecd13d62e53a,closed,chore: split job,saturday06,NaN,2025-04-25T22:09:40Z,2025-04-25T22:24:11Z,2025-04-25T22:24:11Z,human,<NA>,871.0
519,https://github.com/shugo/textbringer/pull/125,159a321e90ee8a9077728bf878cb7b4787e12244,closed,Add copilot-setup-steps.yml for Copilot coding...,shugo,NaN,2025-07-04T07:26:41Z,2025-07-04T07:27:30Z,2025-07-04T07:27:30Z,human,<NA>,49.0
520,https://github.com/tegnike/aituber-kit/pull/402,1d374a47e98fc06ddad98e5e4e4a7e4394a79cf2,closed,Add Claude Code GitHub Workflow,tegnike,## 🤖 Installing Claude Code GitHub App\r\n\r\n...,2025-05-29T15:00:27Z,2025-05-29T15:00:33Z,2025-05-29T15:00:33Z,human,<NA>,6.0
